In [ ]:
from google.colab import files
uploaded = files.upload()




Saving mnist_test.csv to mnist_test (1).csv
Saving mnist_train.csv to mnist_train (1).csv


In [2]:
import copy
import pandas as pd
df_train = pd.read_csv('mnist_train.csv')
X = df_train.drop("label", axis=1).values.astype("float32") / 255.0
Y = df_train["label"].values
print(X.shape)
print(Y.shape)
X_orig = copy.deepcopy(X)
Y_orig = copy.deepcopy(Y)
X = X_orig[:20000]
Y=Y_orig[:20000]

(60000, 784)
(60000,)


In [ ]:
import time
import psutil
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.long)
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(nn.Linear(784, 128),nn.ReLU(),nn.Linear(128, 64),nn.ReLU(),nn.Linear(64, 10))
    def forward(self, x):
        return self.network(x)

#LLM generated code was used for memory calculation in the following codes

model = MNISTNet()
learning_rate = 0.03
num_iterations = 3000
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
process = psutil.Process(os.getpid())
costs = []
memory_usage = []
peak_memory = 0
tic = time.time()

for iteration in range(num_iterations):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, Y)
    loss.backward()
    optimizer.step()
    current_memory = process.memory_info().rss / (1024 ** 2)
    memory_usage.append(current_memory)
    peak_memory = max(peak_memory, current_memory)
    costs.append(loss.item())
    if iteration % 100 == 0:
        print(f"Iteration {iteration:4d} | "
            f"Loss: {loss.item():.6f} | "
            f"Memory: {current_memory:.2f} MB"
        )

toc = time.time()

print("\nTraining Time:", toc - tic, "seconds")
print("Peak Memory Usage:", peak_memory, "MB")
with torch.no_grad():
    outputs = model(X)
    predictions = torch.argmax(outputs, dim=1)
    accuracy = (predictions == Y).float().mean() * 100

print(f"Training Accuracy: {accuracy:.2f}%")

Iteration    0 | Loss: 2.307231 | Memory: 1218.25 MB
Iteration  100 | Loss: 2.174572 | Memory: 1218.58 MB
Iteration  200 | Loss: 1.669484 | Memory: 1218.58 MB
Iteration  300 | Loss: 0.979485 | Memory: 1218.59 MB
Iteration  400 | Loss: 0.664435 | Memory: 1218.59 MB
Iteration  500 | Loss: 0.528827 | Memory: 1218.59 MB
Iteration  600 | Loss: 0.457412 | Memory: 1218.60 MB
Iteration  700 | Loss: 0.414888 | Memory: 1218.60 MB
Iteration  800 | Loss: 0.386538 | Memory: 1218.60 MB
Iteration  900 | Loss: 0.365745 | Memory: 1218.61 MB
Iteration 1000 | Loss: 0.349276 | Memory: 1218.62 MB
Iteration 1100 | Loss: 0.335580 | Memory: 1218.62 MB
Iteration 1200 | Loss: 0.323778 | Memory: 1218.63 MB
Iteration 1300 | Loss: 0.313369 | Memory: 1218.63 MB
Iteration 1400 | Loss: 0.304022 | Memory: 1218.63 MB
Iteration 1500 | Loss: 0.295523 | Memory: 1218.63 MB
Iteration 1600 | Loss: 0.287656 | Memory: 1218.63 MB
Iteration 1700 | Loss: 0.280238 | Memory: 1218.64 MB
Iteration 1800 | Loss: 0.273235 | Memory: 1218

In [4]:
df_test = pd.read_csv('mnist_test.csv')
X_test = df_test.drop("label", axis=1).values.astype("float32") / 255.0
Y_test = df_test["label"].values
X_test = torch.tensor(X_test, dtype=torch.float32)
Y_test = torch.tensor(Y_test, dtype=torch.long)
with torch.no_grad():
    outputs = model(X_test)
    predictions = torch.argmax(outputs, dim=1)
    accuracy1 = (predictions == Y_test).float().mean() * 100
print(f"Test Accuracy: {accuracy1:.2f}%")

Test Accuracy: 93.54%


In [5]:
import numpy as np
def confusion_matrix(y_true, y_pred, num_classes=10):
    cm = np.zeros((num_classes, num_classes), dtype=int)

    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1

    return cm
cm = confusion_matrix(Y_test,predictions,num_classes=10)
print(cm)


[[ 965    0    1    1    0    4    5    2    2    0]
 [   0 1112    2    2    0    1    4    2   12    0]
 [   8    5  960   11    8    0   11   10   15    4]
 [   2    1   24  929    1   19    1   11   17    5]
 [   1    2    4    0  928    0   12    3    4   28]
 [  10    2    3   25    7  797   16    5   21    6]
 [  11    3    4    2   13   10  912    1    2    0]
 [   3    9   24    4    6    1    0  957    1   23]
 [   6    8    3   15    8   22   10    9  879   14]
 [  11    9    1   10   34    7    0   18    4  915]]
